# MALTA-GEO API — Notebook de Testes
Use este notebook para testar os endpoints da API.

In [1]:
import requests
import json
from dotenv import load_dotenv
load_dotenv()
BASE_URL = "http://127.0.0.1:5000"

def pretty(response):
    """Pretty-print a requests.Response."""
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    except Exception:
        print(response.text)

---
## 1. Health check

In [2]:
r = requests.get(f"{BASE_URL}")
print(r.text)

MALTA-GEO API


---
## 2. Chamada simples

In [7]:
payload = {
    "query": "Onde fica a Bacia do Araripe?",
    "session_id" : ""
}
headers = {
    "X-API-Key": "malta_w7-fzdzoRYNMAvGDCyRBpAReYdjrQagj"
}

r = requests.post(f"{BASE_URL}/complete", json=payload, headers=headers)
pretty(r)

Status: 200
{
  "generated_text": "\n\nA Bacia do Araripe está localizada na região Nordeste do Brasil, abrangendo partes dos estados do Ceará, Pernambuco e Piauí. Situa-se no interior do Planalto da Borborema, ocupando uma área de aproximadamente 8.000 km². Sua porção mais conhecida é o Chapadão do Araripe, um relevo tabular que se destaca na paisagem semiárida da região.",
  "session_id": "sess_cXZQhj8xX2ELfZB_C8TjJQ"
}


---
## 3. Chamada com `include_sources` (retorna os chunks usados como contexto)

In [12]:
payload = {
    "query": "Descreva as principais unidades estratigráficas da Bacia do Araripe.",
    "rag_enable": True
}
headers = {
    "X-API-Key": "malta_w7-fzdzoRYNMAvGDCyRBpAReYdjrQagj"
}
r = requests.post(f"{BASE_URL}/complete", json=payload, headers=headers)
pretty(r)



Status: 200
{
  "generated_text": "\n\nA Bacia do Araripe, localizada no Nordeste do Brasil, possui uma estratigrafia dividida em três grupos principais, organizados da base (mais antiga) para o topo (mais recente):\n\n### 1. **Grupo Cariri (Jurássico Superior a Cretáceo Inferior)**  \n   - **Formação Brejo Santo**: Rochas clásticas (conglomerados e arenitos) depositadas em ambientes fluviais e aluviais.  \n   - **Formação Missão Velha**: Arenitos grossos e conglomerados, associados a sistemas fluviais entrelaçados.  \n   - **Formação Abaiara**: Folhelhos e siltitos, indicando ambientes lacustres ou de planície de inundação.  \n\n### 2. **Grupo Araripina (Cretáceo Inferior)**  \n   - **Formação Crato**: Calcários laminados e folhelhos betuminosos, famosos por fósseis excepcionalmente preservados (insetos, plantas e vertebrados) em ambiente lacustre de águas estratificadas.  \n   - **Formação Ipubi**: Evaporitos (gipsita e anidrita) intercalados com folhelhos, refletindo um lago raso co

---
## 4. Chamada com contextos extras (`inner_contexts` / `outer_contexts`)

In [ ]:
payload = {
    "query": "Quais fósseis foram encontrados na Formação Santana?",
    "inner_contexts": ["Formação Santana", "Cretáceo"],
    "outer_contexts": ["paleontologia", "peixes fósseis"],
   # "include_sources": True
}

r = requests.post(f"{BASE_URL}/complete", json=payload)
pretty(r)

---
## 5. Teste de múltiplas perguntas em lote

In [ ]:
perguntas = [
    "Onde fica a Bacia do Araripe?",
    "Quais são as formações geológicas da Bacia do Araripe?",
    "Qual a importância paleontológica da Bacia do Araripe?",
    "O que é a Formação Crato?",
    "Quais minerais são encontrados na Bacia do Araripe?",
]

for pergunta in perguntas:
    print(f"\n{'='*60}")
    print(f"PERGUNTA: {pergunta}")

    r = requests.post(f"{BASE_URL}/complete", json={"query": pergunta})

    if r.status_code == 200:
        data = r.json()
        print(data.get("generated_text", data))
    else:
        print(f"ERRO {r.status_code}: {r.text}")

---
## 6. Pergunta fora do escopo (deve recusar)

In [ ]:
payload = {
    "query": "Qual a capital da França?"
}

r = requests.post(f"{BASE_URL}/complete", json=payload)
pretty(r)

---
## 7. Chamada interativa — digite sua própria pergunta

In [ ]:
pergunta = input("Digite sua pergunta: ")

r = requests.post(
    f"{BASE_URL}/complete",
    json={"query": pergunta, "include_sources": True}
)

data = r.json()

print("\n=== RESPOSTA ===")
print(data.get("generated_text", data))

print("\n=== FONTES ===")
for i, src in enumerate(data.get("sources", []), 1):
    print(f"\n[{i}] {src['metadata'].get('file_name', '')}")
    print(src['content'][:300], "...")